In [83]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

In [84]:
df_raw = pd.read_csv('IndustryPrices.csv')

In [85]:
df_raw.tail()

,Date,AAPL,MSFT,AMZN,GOOGL,META,ORCL,CRM,ADBE,IBM,...,SRE,ED,XEL,PCG,EIX,FE,CMS,DTE,CNP,ES
2512,2025-03-31,222.1300048828125,375.3900146484375,190.25999450683594,154.63999938964844,576.3599853515625,139.80999755859375,268.3599853515625,383.5299987792969,248.66000366210938,...,71.36000061035156,110.58999633789062,70.79000091552734,17.18000030517578,58.91999816894531,40.41999816894531,75.11000061035156,138.27000427246094,36.22999954223633,62.11000061035156
2513,2025-04-01,223.19000244140625,382.19000244140625,192.1699981689453,157.07000732421875,586.0,141.94000244140625,270.20001220703125,383.20001220703125,250.33999633789062,...,71.55999755859375,110.05999755859375,70.72000122070312,17.270000457763672,58.75,40.400001525878906,75.13999938964844,137.9199981689453,36.79999923706055,62.09000015258789
2514,2025-04-02,223.88999938964844,382.1400146484375,196.00999450683594,157.0399932861328,583.9299926757812,145.86000061035156,271.5400085449219,385.7799987792969,249.97999572753906,...,72.83999633789062,109.58000183105469,70.9000015258789,17.43000030517578,59.91999816894531,40.58000183105469,74.66000366210938,138.0,36.93000030517578,62.630001068115234
2515,2025-04-03,203.19000244140625,373.1099853515625,178.41000366210938,150.72000122070312,531.6199951171875,137.22999572753906,255.22999572753906,367.25,243.49000549316406,...,70.7300033569336,112.72000122070312,72.13999938964844,17.25,58.380001068115234,41.04999923706055,75.51000213623047,139.49000549316406,37.36000061035156,61.91999816894531
2516,Industry,Technology,Technology,Technology,Technology,Technology,Technology,Technology,Technology,Technology,...,Utilities,Utilities,Utilities,Utilities,Utilities,Utilities,Utilities,Utilities,Utilities,Utilities


In [86]:
#get list of different industries
sector_key = df_raw.iloc[-1]

sectors_list = df_raw.iloc[-1].dropna().unique().tolist()
if 'Date' in sectors_list:
    sectors_list.remove('Date')

In [87]:
sectors_list

['Industry',
 'Technology',
 'Financial Services',
 'Consumer Cyclical',
 'Healthcare',
 'Communication Services',
 'Industrials',
 'Consumer Defensive',
 'Energy',
 'Real Estate',
 'Basic Materials',
 'Utilities']

In [88]:
# Make dictionary of every company in each industry
df_sectors = {}

for i in sectors_list:
    df_sectors[f'{i}'] = sector_key[sector_key == f'{i}'].index.to_list()
    
    # Drop "Date" if it exists in the index
    if 'Date' in df_sectors[f'{i}']:
        df_sectors[f'{i}'].remove('Date')  # Use remove() since it's a list

In [89]:
# Example
df_sectors['Technology']

['AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'META',
 'ORCL',
 'CRM',
 'ADBE',
 'IBM',
 'INTC',
 'NVDA',
 'CSCO']

In [90]:
# Calculate weekly average and weekly return for each industry

df_averages = df_raw.copy()

# Convert columns to floats
for col in df_averages.columns:
    if col != 'Date':
        df_averages[col] = pd.to_numeric(df_averages[col], errors='coerce')


df_averages = df_averages.iloc[:-1]

# Assuming df_raw is your raw data
date_col = 'Date'  # Change this to match your actual date column name

# Apply `pd.to_numeric()` only to non-date columns
df_averages.loc[:, df_averages.columns != date_col] = df_averages.loc[:, df_averages.columns != date_col].apply(pd.to_numeric, errors='coerce')

for sector in sectors_list:
    # Now, you can calculate the average safely
    sector_mask = df_sectors[f'{sector}']
    
    # Compute the average of 'df_raw' for rows where the 'TECH' sector is True
    df_averages[f'{sector}_avg'] = df_averages[sector_mask].mean(axis=1)
    
    #Return average
    df_averages[f'{sector}_wk_pct_change'] = df_averages[f'{sector}_avg'].pct_change()

C:\Users\etien\AppData\Local\Temp\ipykernel_36600\3805325079.py:24: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\etien\AppData\Local\Temp\ipykernel_36600\3805325079.py:27: FutureWarning:

The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.

C:\Users\etien\AppData\Local\Temp\ipykernel_36600\3805325079.py:27: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.co

In [91]:
df_averages.tail()

,Date,AAPL,MSFT,AMZN,GOOGL,META,ORCL,CRM,ADBE,IBM,...,Consumer Defensive_avg,Consumer Defensive_wk_pct_change,Energy_avg,Energy_wk_pct_change,Real Estate_avg,Real Estate_wk_pct_change,Basic Materials_avg,Basic Materials_wk_pct_change,Utilities_avg,Utilities_wk_pct_change
2511,2025-03-28,217.899994,378.799988,192.720001,154.330002,576.739990,140.869995,269.970001,385.709991,244.000000,...,143.787058,-0.004318,103.476667,-0.008615,167.538467,-0.007543,162.324719,-0.016700,72.450313,0.010610
2512,2025-03-31,222.130005,375.390015,190.259995,154.639999,576.359985,139.809998,268.359985,383.529999,248.660004,...,145.959414,0.015108,104.625000,0.011098,169.661333,0.012671,164.325716,0.012327,73.575625,0.015532
2513,2025-04-01,223.190002,382.190002,192.169998,157.070007,586.000000,141.940002,270.200012,383.200012,250.339996,...,146.524707,0.003873,105.350833,0.006937,169.999998,0.001996,164.918573,0.003608,73.441875,-0.001818
2514,2025-04-02,223.889999,382.140015,196.009995,157.039993,583.929993,145.860001,271.540009,385.779999,249.979996,...,146.625882,0.000690,105.381666,0.000293,171.533998,0.009024,166.330000,0.008558,73.474376,0.000443
2515,2025-04-03,203.190002,373.109985,178.410004,150.720001,531.619995,137.229996,255.229996,367.250000,243.490005,...,148.226472,0.010916,95.495000,-0.093818,165.170664,-0.037097,160.607857,-0.034402,74.243126,0.010463


In [92]:
# Plot the weekly value of every industry

col_averages_weekly = [col for col in df_averages.columns if '_avg' in col]

fig = px.line(df_averages, x='Date', y=col_averages_weekly)
fig.show()

In [93]:
# Plot the weekly portfolio change for every industry

col_averages_returns = [col for col in df_averages.columns if '_rtrn' in col]

fig = px.line(df_averages, x='Date', y=col_averages_returns)
fig.show()

Plot stocks in a particular group

In [95]:
df_sectors['Technology']

['AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'META',
 'ORCL',
 'CRM',
 'ADBE',
 'IBM',
 'INTC',
 'NVDA',
 'CSCO']

In [96]:
# Plots the individual stocks for a specific industry
def plot_stocks_for_industry(df, industry):
    col_names = df_sectors[industry]

    fig = px.line(df, x='Date', y=col_names)
    return fig

In [97]:
df_averages.to_csv('df_cleaned.csv')

In [98]:
# Chart for a specific industry
plot_stocks_for_industry(df_averages, 'Technology')

In [99]:
plot_stocks_for_industry(df_averages, 'Utilities')

In [100]:
# Dash 
app = Dash(__name__)

app.layout = html.Div([
    html.H4('Stock price analysis'),
    dcc.Graph(id="time-series-chart"),
    html.P("Select stock:"),
    dcc.Dropdown(
        id="ticker",
        options=sorted(df_sectors['Technology']),
        value="AMZN",
        clearable=False,
    ),
])


@app.callback(
    Output("time-series-chart", "figure"),
    Input("ticker", "value"))

def display_time_series(ticker):
    df = df_averages # replace with your own data source
    fig = px.line(df, x='Date', y=ticker)
    return fig

app.run(debug=True)

In [101]:
#define risk
#define goal
#focus on a specific industry (why just the top 12? Are those really the best stocks?)
#high volume traded stocks
#long-term or daily trading focus
#goal to beat the index or win in short term
#present different portfolio options (read articles on it)
#or can compare different industries and their trends